<div dir="rtl" align="right">

# مُرشِّحُ النطاقِ في الوقتِ الحقيقيِّ - البثُّ مقابل المعالجةِ دون اتصالٍ

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يُوضّحُ هذا الدفترُ **ترشيحَ النطاقِ في الوقتِ الحقيقيِّ** مع البياناتِ المُتدفّقةِ. نُصمّمُ مُرشِّحَ Butterworth للنطاقِ (0.5-30 Hz، رتبةٌ 4) ونُطبّقُه باستخدامِ `scipy.signal.lfilter` مع الحفاظِ على الحالةِ بين الحزمِ. نُقارنُ هذا بنهجِ `filtfilt` دون اتصالٍ، الذي يتطلّبُ الإشارةَ كاملةً.

## ماذا يَفعلُ هذا الدفترُ

1. يحمّلُ قناةَ P4 من مجموعةِ بياناتِ EEG المحليةِ
2. يُصمّمُ مُرشِّحَ Butterworth للنطاقِ (0.5-30 Hz، رتبةٌ 4)
3. يُعالجُ الإشارةَ في حزمٍ من 50 عينةٍ باستخدامِ `lfilter` مع الحفاظِ على الحالةِ `zi`
4. يُحسبُ النسخةَ المُرشَّحةَ دون اتصالٍ باستخدامِ `filtfilt` للمقارنةِ
5. يَرسمُ النتيجتينِ لإظهارِ الفرقِ

## المُخرجاتُ المُتوقّعةُ

- المُخطّطُ العلويُّ يُظهرُ **الإشارةَ الخامَ الأصليةَ** (أولُ 5000 عينةٍ) باللونِ الأزرقِ
- المُخطّطُ السفليُّ يُظهرُ **الإشارةَ المُرشَّحةَ في الوقتِ الحقيقيِّ** باللونِ الأخضرِ (باستخدامِ lfilter مع الحالةِ)
- **الإشارةُ المُرشَّحةُ دون اتصالٍ** مُرسمةٌ فوقها باللونِ الأحمرِ المتقطّعِ للمقارنةِ
- المُرشِّحُ في الوقتِ الحقيقيِّ له عبورٌ خفيفٌ في البدايةِ لكنه يَتقاربُ ليتطابقَ مع النسخةِ دون اتصالٍ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | الوصفُ |
| --------- | ------- | ------ |
| FS | 200 Hz | معدّلُ أخذِ العيناتِ |
| LOWCUT | 0.5 Hz | حدُّ النطاقِ المنخفضُ |
| HIGHCUT | 30.0 Hz | حدُّ النطاقِ العاليُ |
| ORDER | 4 | رتبةُ مُرشِّحِ Butterworth |
| CHUNK_SIZE | 50 | العيناتُ في كلِّ حزمةِ معالجةٍ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb

<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

<div dir="rtl" align="right">

## 4. تطبيقُ المعالجةِ

نُصمّمُ مُرشِّحَ Butterworth للنطاقِ (0.5-30 Hz، رتبةٌ 4) ونُطبّقُه بطريقتينِ:
1. **في الوقتِ الحقيقيِّ** باستخدامِ `lfilter` مع الحفاظِ على الحالةِ `zi` بين حزمٍ من 50 عينةٍ
2. **دون اتصالٍ** باستخدامِ `filtfilt` للترشيحِ ذي الطورِ الصفريِّ (يتطلّبُ الإشارةَ كاملةً)

</div>

In [ ]:
from scipy import signal

LOWCUT = 0.5
HIGHCUT = 30.0
ORDER = 4
CHUNK_SIZE = 50
N_PLOT = 5000

n_samples = min(N_PLOT, len(channel_data))
signal_plot = channel_data[:n_samples]

nyq = 0.5 * fs
b, a = signal.butter(ORDER, [LOWCUT / nyq, HIGHCUT / nyq], btype='band', analog=False)

zi = signal.lfilter_zi(b, a)
state = zi * signal_plot[0]
realtime_filtered = np.zeros(n_samples)

for start in range(0, n_samples, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n_samples)
    chunk = signal_plot[start:end]
    filtered_chunk, state = signal.lfilter(b, a, chunk, zi=state)
    realtime_filtered[start:end] = filtered_chunk

offline_filtered = signal.filtfilt(b, a, signal_plot)

print(f'Bandpass: {LOWCUT}-{HIGHCUT} Hz, order {ORDER}')
print(f'Real-time filtered {n_samples} samples in chunks of {CHUNK_SIZE}')
print(f'Offline filtered using filtfilt (zero-phase)')

<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**
- المُخطّطُ العلويُّ يُظهرُ الإشارةَ الخامَ بكلِّ ضجيجها وانجرافِها
- المُخطّطُ السفليُّ يُظهرُ الإشارةَ المُرشَّحةَ في الوقتِ الحقيقيِّ (أخضرُ) والإشارةَ المُرشَّحةَ دون اتصالٍ (أحمرُ متقطّعٌ)
- للمُرشِّحِ في الوقتِ الحقيقيِّ **عبورٌ خفيفٌ** في البدايةِ (أولُ عيناتٍ) بسبب تهيئةِ المُرشِّحِ
- بعد العبورِ، **تَتقاربُ** الإشارتانِ وتبدوانِ مُتطابقتينِ تقريبًا
- `filtfilt` دون اتصالٍ له طورٌ صفريٌّ، بينما يُحدثُ `lfilter` إزاحةَ طورٍ صغيرةً

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

x = np.arange(n_samples)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original Signal (P4)',
                                    f'Bandpass Filtered ({LOWCUT}-{HIGHCUT} Hz)'))

fig.add_trace(go.Scatter(x=x, y=signal_plot, name='Original',
                         line=dict(color='blue', width=0.5)), row=1, col=1)

fig.add_trace(go.Scatter(x=x, y=realtime_filtered, name='Real-time (lfilter)',
                         line=dict(color='green', width=0.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=x, y=offline_filtered, name='Offline (filtfilt)',
                         line=dict(color='red', width=0.5, dash='dash')), row=2, col=1)

fig.update_layout(height=600, title_text='Real-time Bandpass Filter - Streaming vs Offline',
                  xaxis2_title='Sample index', yaxis_title='Amplitude (uV)',
                  yaxis2_title='Amplitude (uV)')
fig.show()

<div dir="rtl" align="right">

## خلاصةٌ

- **`lfilter`** يُعالجُ البياناتَ بالتتابعِ ويُحافظُ على **حالةِ المُرشِّحِ** (`zi`) بين الحزمِ
- **`filtfilt`** يُطبّقُ المُرشِّحَ للأمامِ والخلفِ للحصولِ على طورٍ صفريٍّ، لكنه يتطلّبُ **الإشارةَ كاملةً**
- المُعاملُ `zi` في `lfilter` يَحملُ الحالةَ الداخليةَ للمُرشِّحِ عبر حدودِ الحزمِ
- يَظهرُ **عبورٌ خفيفٌ** في بدايةِ الترشيحِ في الوقتِ الحقيقيِّ، لكنه يَتقاربُ بسرعةٍ
- الترشيحُ في الوقتِ الحقيقيِّ يُحدثُ **إزاحةَ طورٍ** صغيرةً يتجنّبها الترشيحُ دون اتصالٍ
- هذا النهجُ ضروريٌّ لـ **أنظمةِ BCI عبر الإنترنتِ** التي يجبُ أن تُعالجَ البياناتِ عند وصولِها

</div>